# E-commerce Customer Retention Analysis

**Portfolio case study using the UCI Online Retail dataset**  
**Source:** UCI Machine Learning Repository — Online Retail, dataset 352.  
**Data:** Real historical transactions from a UK-based online retailer (December 2010–December 2011).  
**Purpose:** Demonstrate transaction cleaning, revenue analysis, customer behaviour, repeat purchasing, and an explicitly labelled inactivity proxy.

> This dataset contains transaction records, not verified customer churn labels. This notebook does not claim to measure contractual churn or prove why customers stopped purchasing. The inactivity analysis is a descriptive proxy based on the dataset's observation window.

## 1. Business questions

1. How much revenue is represented in the transaction data over time?
2. Which products and countries contribute most to recorded revenue?
3. How many customers make repeat purchases?
4. How does customer purchasing activity vary by cohort?
5. Which customers have no recorded purchase near the end of the observation period?

The last question is an **inactivity proxy**, not a confirmed churn label. A customer may have purchased elsewhere, returned later, or simply not needed another order.

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()
archive = zipfile.ZipFile(io.BytesIO(response.content))
excel_files = [name for name in archive.namelist() if name.lower().endswith('.xlsx')]
if not excel_files:
    raise FileNotFoundError('No Excel workbook found in the downloaded UCI archive.')
print('Workbook:', excel_files[0])
df_raw = pd.read_excel(archive.open(excel_files[0]))
df_raw.head()

## 2. Data dictionary and audit

The source includes invoice, product description, quantity, invoice date, unit price, customer identifier, and country. Inspect the actual imported columns and quality before transforming the data.

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumns:', df_raw.columns.tolist())
print('\nDtypes:\n', df_raw.dtypes)
print('\nMissing values:\n', df_raw.isna().sum().sort_values(ascending=False))
print('\nDuplicate rows:', df_raw.duplicated().sum())
display(df_raw.describe(include='all').T)

## 3. Cleaning and feature engineering

Cancellation invoices are excluded from positive sales analysis. Rows with missing customer IDs are excluded from customer-level retention metrics but can remain relevant to overall sales totals. Negative quantities and non-positive prices are excluded from this positive-sales analysis; document this choice when interpreting results.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().replace(' ', '') for c in df.columns]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['CustomerID'] = df['CustomerID'].astype('string')
df['Revenue'] = df['Quantity'] * df['UnitPrice']

audit = {
 'raw_rows': len(df),
 'cancellation_invoice_rows': int(df['InvoiceNo'].astype(str).str.startswith('C').sum()),
 'missing_customer_id_rows': int(df['CustomerID'].isna().sum()),
 'non_positive_quantity_rows': int((df['Quantity'] <= 0).sum()),
 'non_positive_price_rows': int((df['UnitPrice'] <= 0).sum()),
 'duplicate_rows': int(df.duplicated().sum())
}
pd.Series(audit, name='count').to_frame()

In [ ]:
sales = df.loc[
    (~df['InvoiceNo'].astype(str).str.startswith('C')) &
    df['CustomerID'].notna() &
    df['InvoiceDate'].notna() &
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()
sales['CustomerID'] = sales['CustomerID'].astype(str).str.replace(r'\.0$', '', regex=True)
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
sales['InvoiceMonth'] = sales['InvoiceDate'].dt.to_period('M')
print('Clean customer transaction rows:', len(sales))
print('Unique customers:', sales['CustomerID'].nunique())
print('Date range:', sales['InvoiceDate'].min(), 'to', sales['InvoiceDate'].max())
display(sales.head())

## 4. Revenue and sales trends

Revenue here is calculated as quantity multiplied by unit price on retained positive-quantity rows. It is not profit and does not account for costs, taxes, shipping, or refunds beyond the filtering rules above.

In [ ]:
monthly = sales.groupby('InvoiceMonth').agg(
    revenue=('Revenue','sum'),
    invoices=('InvoiceNo','nunique'),
    customers=('CustomerID','nunique')
).reset_index()
monthly['InvoiceMonth'] = monthly['InvoiceMonth'].astype(str)
display(monthly.head())

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(monthly['InvoiceMonth'], monthly['revenue'], marker='o')
ax.set_title('Recorded monthly sales revenue')
ax.set_xlabel('Invoice month')
ax.set_ylabel('Revenue (dataset currency: GBP)')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

In [ ]:
top_products = (sales.groupby('Description', dropna=True)['Revenue']
                .sum().sort_values(ascending=False).head(10))
top_countries = (sales.groupby('Country')['Revenue']
                 .sum().sort_values(ascending=False).head(10))
display(top_products.to_frame('revenue'))
display(top_countries.to_frame('revenue'))

fig, ax = plt.subplots(figsize=(9,5))
top_countries.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 countries by recorded revenue')
ax.set_xlabel('Revenue (GBP)')
plt.tight_layout(); plt.show()

## 5. Customer repeat purchasing

A repeat customer is defined here as a customer with more than one distinct invoice in the observation period. This is a descriptive definition and should not be confused with a formal retention rate.

In [ ]:
customer_orders = sales.groupby('CustomerID').agg(
    invoice_count=('InvoiceNo','nunique'),
    order_lines=('InvoiceNo','size'),
    total_revenue=('Revenue','sum'),
    first_purchase=('InvoiceDate','min'),
    last_purchase=('InvoiceDate','max')
).reset_index()
customer_orders['repeat_customer'] = customer_orders['invoice_count'] > 1
repeat_summary = pd.Series({
    'unique_customers': customer_orders['CustomerID'].nunique(),
    'customers_with_multiple_invoices': int(customer_orders['repeat_customer'].sum()),
    'repeat_customer_share': customer_orders['repeat_customer'].mean(),
    'median_invoices_per_customer': customer_orders['invoice_count'].median()
})
display(repeat_summary.to_frame('value'))
display(customer_orders.sort_values('total_revenue', ascending=False).head(10))

## 6. Cohort analysis

Cohorts are based on each customer's first recorded purchase month. Monthly activity is measured as the share of cohort customers with at least one invoice in each subsequent month represented in the dataset. The final columns may be incomplete because the dataset ends in December 2011.

In [ ]:
cohort_data = sales[['CustomerID','InvoiceDate','InvoiceNo']].copy()
cohort_data['InvoiceMonth'] = cohort_data['InvoiceDate'].dt.to_period('M')
cohort_data['CohortMonth'] = cohort_data.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')
cohort_data['CohortIndex'] = (
    (cohort_data['InvoiceMonth'].dt.year - cohort_data['CohortMonth'].dt.year) * 12 +
    (cohort_data['InvoiceMonth'].dt.month - cohort_data['CohortMonth'].dt.month) + 1
)
cohort_counts = cohort_data.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().unstack(1)
cohort_sizes = cohort_counts.iloc[:,0]
retention = cohort_counts.divide(cohort_sizes, axis=0)
retention.index = retention.index.astype(str)
display(retention.style.format('{:.0%}').background_gradient(cmap='Blues', axis=None))

## 7. Inactivity proxy — not confirmed churn

For a descriptive snapshot, define the reference date as one day after the latest transaction in the dataset. Flag customers whose last recorded purchase was more than 90 days before that date. This only indicates absence of a recorded purchase during the remaining observation window; it does not prove permanent churn.

In [ ]:
reference_date = sales['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
customer_orders['days_since_last_purchase'] = (reference_date - customer_orders['last_purchase'].dt.normalize()).dt.days
customer_orders['inactive_90d_proxy'] = customer_orders['days_since_last_purchase'] > 90
proxy_summary = pd.Series({
    'reference_date': str(reference_date.date()),
    'customers_in_scope': len(customer_orders),
    'customers_over_90d_inactive_proxy': int(customer_orders['inactive_90d_proxy'].sum()),
    'share_over_90d_inactive_proxy': customer_orders['inactive_90d_proxy'].mean()
})
display(proxy_summary.to_frame('value'))
display(customer_orders.sort_values('days_since_last_purchase', ascending=False).head(10))

## 8. Conclusions and recommendations

Run the notebook before writing numerical conclusions. Use the resulting tables and charts to complete the following evidence-based summary:
- **Revenue trend:** Identify the strongest and weakest observed months and investigate seasonality.
- **Customer mix:** Compare country-level revenue with customer counts; high revenue does not necessarily mean high customer volume.
- **Repeat purchasing:** Report the measured share of customers with multiple invoices, clearly stating the definition.
- **Cohort behaviour:** Compare cohort activity while noting that later cohorts have shorter observation windows.
- **Inactivity proxy:** Treat the 90-day flag as a screening signal only. It is not a confirmed churn label.

Potential business actions to investigate with the retailer include repeat-purchase campaigns, product assortment review, and targeted customer re-engagement. These are hypotheses, not proven interventions.

## 9. Limitations and reproducibility

- The dataset is historical and represents one retailer; it is not representative of all e-commerce businesses.
- Customer IDs are missing for some transaction rows; those rows are excluded from customer-level metrics.
- Positive-sales filtering excludes cancellations, negative quantities, and non-positive unit prices from this analysis.
- Revenue is a transaction-value calculation, not profit.
- Cohort and inactivity measures are bounded by the dataset's observation window.
- The 90-day inactivity proxy is not a verified churn outcome.
- This analysis is observational and cannot establish causation.

**Source:** Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33  
**License:** Check the current UCI dataset page and terms before redistribution or commercial reuse. This notebook downloads the dataset at runtime rather than committing the source data file.